读取数据

In [6]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw/"
OUTPUT_DIR = "../data/processed/"

wage = pd.read_excel(
    DATA_DIR + "工资水平.xlsx",
    sheet_name="职工平均工资"
)

wage.head()
#wage.shape

,averageWage,city1,city2,city3,city4,city5,city6,city7,city8,city9,...,city31,city32,city33,city34,city35,city36,city37,city38,city39,city40
0,2001年,10340.0,11984,23036.0,11504,11776,12496.0,11460.0,7976,8216,...,9100.0,9300,12496,7736,7080.0,6872,9732,8988.0,15328,10276
1,2002年,13044.0,13780,25940.0,14732,13668,14220.0,14728.0,8948,9472,...,10260.0,10532,15172,8608,9516.0,7668,10760,9668.0,17684,11740
2,2003年,16648.0,15796,28216.0,16688,15920,15872.0,16860.0,9852,10380,...,11168.0,12020,17240,9688,10516.0,9188,12004,11184.0,19172,13192
3,2004年,19968.0,18928,30608.0,18776,19788,17640.0,19692.0,10800,12436,...,12404.0,13400,18912,10616,12032.0,10508,13800,13068.0,21180,15260
4,2005年,22024.0,22124,31928.0,22248,22508,19560.0,21576.0,12168,14528,...,13736.0,15080,19576,11948,13580.0,12868,15732,14912.0,24264,17380


宽表转长表，这样才可匹配其他csv

In [7]:
wage_long = wage.melt(
    id_vars=["averageWage"],
    var_name="city_id",
    value_name="average_wage"
)

wage_long.head()

,averageWage,city_id,average_wage
0,2001年,city1,10340.0
1,2002年,city1,13044.0
2,2003年,city1,16648.0
3,2004年,city1,19968.0
4,2005年,city1,22024.0


In [8]:
wage_long["year"] = (
    wage_long["averageWage"]
    .astype(str)
    .str.replace("年", "", regex=False)
)

wage_long[["averageWage", "year"]].head()


,averageWage,year
0,2001年,2001
1,2002年,2002
2,2003年,2003
3,2004年,2004
4,2005年,2005


将字符串转为实际数值，删除原年份col，统一其他表，最后检查

In [9]:
wage_long["year"] = (
    wage_long["averageWage"]
    .astype(str)
    .str.replace("年", "", regex=False)
)

wage_long["year"] = pd.to_numeric(
    wage_long["year"],
    errors="coerce"
)

wage_long = wage_long.drop(
    columns=["averageWage"]
)

wage_long = wage_long[
    ["city_id", "year", "average_wage"]
]

print("城市数量：", wage_long["city_id"].nunique())
print("最早年份：", wage_long["year"].min())
print("最晚年份：", wage_long["year"].max())

sorted(
    wage_long["city_id"].unique(),
    key=lambda x: int(x.replace("city", ""))
)

城市数量： 40
最早年份： 2001
最晚年份： 2021


['city1',
 'city2',
 'city3',
 'city4',
 'city5',
 'city6',
 'city7',
 'city8',
 'city9',
 'city10',
 'city11',
 'city12',
 'city13',
 'city14',
 'city15',
 'city16',
 'city17',
 'city18',
 'city19',
 'city20',
 'city21',
 'city22',
 'city23',
 'city24',
 'city25',
 'city26',
 'city27',
 'city28',
 'city29',
 'city30',
 'city31',
 'city32',
 'city33',
 'city34',
 'city35',
 'city36',
 'city37',
 'city38',
 'city39',
 'city40']

质量报告建立

In [10]:
def quality_report(df, name):
    print("=" * 60)
    print(name)

    print("Shape：", df.shape)
    print("城市数量：", df["city_id"].nunique())
    print("最早年份：", df["year"].min())
    print("最晚年份：", df["year"].max())

    print(
        "重复 city-year：",
        df.duplicated(["city_id", "year"]).sum()
    )

    print("各字段缺失值：")
    print(df.isna().sum())
    
quality_report(
    wage_long,
    "职工平均工资"
)


职工平均工资
Shape： (840, 3)
城市数量： 40
最早年份： 2001
最晚年份： 2021
重复 city-year： 0
各字段缺失值：
city_id          0
year             0
average_wage    49
dtype: int64


In [11]:
wage_long.to_csv(
    OUTPUT_DIR + "wage_clean.csv",
    index=False
)